In [ ]:
import sys
import os

import itertools
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm as tqdm_auto
from tqdm.notebook import tqdm

import torch
from torch.utils.data import DataLoader, Dataset
import pytorch_lightning as pl
from IPython.display import clear_output


mpl.rcParams['pdf.fonttype'] = 42

In [ ]:

from utilities.pl_regressor import RNARegressor
from utilities.UTR3_utilities import predict, seqprep, Condition2Tensor, UTRData, make_res


In [ ]:
from utilities.legnet_classifier import LegNetClassifier
import utilities.legnet_classifier as legnet_classifier
sys.modules['legnet_classifier'] = legnet_classifier

In [ ]:
def seed_everything(seed):
    # random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    
seed_everything(42)

In [ ]:
device = 'cuda:3'
num_workers = 16

batch_size = 64
seq_len = 240
CELL_TYPE_FILTER = 'c2'

CELLTYPE_CODES_UTR3 = {"c1": 0,
                       "c2": 1,
                       "c4": 2,
                       "c6": 3,
                       "c17": 4,
                       "c13": 5,
                       "c10": 6}

CELLTYPE_UTR3 = {"c1": 0,
                       "c2": 1,
                       "c4": 2,
                       "c6": 3,
                       "c17": 4,
                       "c13": 5}

CELLTYPE_CODES_UTR5 = {"c1": 0,
                       "c2": 1,
                       "c4": 2,
                       "c6": 3,
                       "c17": 4}


c2t = Condition2Tensor(num_conditions=len(CELLTYPE_CODES_UTR3), celltype_codes=CELLTYPE_CODES_UTR3)

In [ ]:
model_path = f"utilities/model-utr3-deltas-epoch=9-step=1330.ckpt"
device = 'cuda:0'

model_predict = RNARegressor.load_from_checkpoint(model_path, map_location='cpu').model.to(device)
model_predict.eval()
input_len = 240

scoring 

In [ ]:
import pandas as pd
import torch
from tqdm import tqdm

in_file = 'data/gan_generated_sequences_UTR3_10k.csv'
cell_lines = ['c1', 'c2', 'c4', 'c6', 'c13', 'c17']
CHUNK_SIZE = 10_000 
BATCH_SIZE = 1024      
MAX_SAMPLES = 1_000_000

for cell_line in cell_lines:
    first = True
    out_file_csv = f'data/scored_UTR3_80dim_10k_{cell_line}.csv'
    df = pd.read_csv(in_file).iloc[:MAX_SAMPLES]
    num_batches = (len(df) + BATCH_SIZE - 1) // BATCH_SIZE

    with tqdm(total=num_batches, desc=f"{cell_line}") as pbar:
        for start in range(0, len(df), CHUNK_SIZE):
            end = min(start + CHUNK_SIZE, len(df))
            chunk = df.iloc[start:end].copy()

            scores_all = []
            for b_start in range(0, len(chunk), BATCH_SIZE):
                b_end = min(b_start + BATCH_SIZE, len(chunk))
                batch = chunk.iloc[b_start:b_end]

                seq_logits = [seqprep(seq) for seq in batch.sequence]
                seq_logits = torch.concat(seq_logits)

                preds = predict(seq_logits, model_predict, cell_line, device)
                scores_all.extend(preds)

                pbar.update(1)

            chunk["score"] = scores_all
            chunk.to_csv(out_file_csv, mode="a", header=first, index=False)
            first = False  

    print(f"[+] Scoring saved to {out_file_csv}")
